In [ ]:
#Models in this file are DeepSeek 2 and Chandra 2

In [2]:
import os
import pandas as pd

DATASET_DIR = "/kaggle/input/datasets/manyabhatnagar/inputs"

CSV_PATH = os.path.join(DATASET_DIR, "ClinOCR_Dataset.csv")
IMAGE_DIR = os.path.join(DATASET_DIR, "ClinOCR_images")

print("CSV exists:", os.path.exists(CSV_PATH))
print("Image directory exists:", os.path.exists(IMAGE_DIR))
print("Number of images:", len(os.listdir(IMAGE_DIR)))

CSV exists: True
Image directory exists: True
Number of images: 192


In [4]:
DATASET_DIR = "/kaggle/input/datasets/manyabhatnagar/inputs"
IMAGE_DIR = os.path.join(DATASET_DIR, "ClinOCR_images")
CSV_PATH = os.path.join(DATASET_DIR, "ClinOCR_Dataset.csv")

df = pd.read_csv(CSV_PATH)

# Convert CSV's relative image path to actual Kaggle path
df["image"] = df["image"].apply(
    lambda x: os.path.join(IMAGE_DIR, os.path.basename(x))
)

print("Rows:", len(df))
print("Images found:", df["image"].apply(os.path.exists).sum())
print("Images missing:", (~df["image"].apply(os.path.exists)).sum())

display(df[["doc_id", "image", "ground_truth"]].head())

Rows: 192
Images found: 192
Images missing: 0


,doc_id,image,ground_truth
0,normal_t1_s1,/kaggle/input/datasets/manyabhatnagar/inputs/C...,"Patient Name: Doe, Jane Patient ID: JG05141985..."
1,normal_t2_s1,/kaggle/input/datasets/manyabhatnagar/inputs/C...,Lakewood Hospital - Pathology Pathology Report...
2,normal_t3_s1,/kaggle/input/datasets/manyabhatnagar/inputs/C...,Riverside Hospital. 742 Evergreen Terrace Spri...
3,normal_t4_s1,/kaggle/input/datasets/manyabhatnagar/inputs/C...,GENERALHOSPITAL\nHOSPITAL\nDate of Visit: 06/2...
4,normal_t5_s1,/kaggle/input/datasets/manyabhatnagar/inputs/C...,"University Medical Center 456 Oak Avenue, Spri..."


In [ ]:
!pip install -q addict

In [ ]:
!pip install -q "transformers==4.46.3"

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
from transformers import AutoModel, AutoTokenizer
print("Imports successful!")

In [ ]:
import huggingface_hub
import httpx

print("huggingface_hub:", huggingface_hub.__version__)
print("httpx:", httpx.__version__)

In [ ]:
from transformers import AutoConfig
from accelerate import init_empty_weights, infer_auto_device_map

MODEL_NAME = "deepseek-ai/DeepSeek-OCR-2"

# --------------------------------------------------
# 1. Load tokenizer + config
# --------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Config loaded")
print(type(config))

In [ ]:
device_map = {
    # -------------------------
    # GPU 0 — vision side
    # -------------------------
    "model.view_seperator": 0,
    "model.embed_tokens": 0,
    "model.sam_model": 0,
    "model.qwen2_model": 0,
    "model.projector": 0,

    # -------------------------
    # GPU 0 — language layers
    # -------------------------
    "model.layers.0": 0,
    "model.layers.1": 0,
    "model.layers.2": 0,
    "model.layers.3": 0,
    "model.layers.4": 0,
    "model.layers.5": 0,

    # -------------------------
    # GPU 1 — language layers
    # -------------------------
    "model.layers.6": 1,
    "model.layers.7": 1,
    "model.layers.8": 1,
    "model.layers.9": 1,
    "model.layers.10": 1,
    "model.layers.11": 1,

    # -------------------------
    # GPU 1 — output
    # -------------------------
    "model.norm": 1,
    "lm_head": 1,
}

In [ ]:
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_safetensors=True,
    device_map=device_map,
    torch_dtype=torch.bfloat16,
)

model.eval()

print("MODEL LOADED")

In [ ]:
import os
import re
import time
import warnings
import io
from contextlib import redirect_stdout, redirect_stderr

import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


def clean_deepseek_output(text):

    if text is None:
        return ""

    text = str(text)

    text = re.sub(
        r"<\|ref\|>.*?<\|/ref\|>\s*"
        r"<\|det\|>.*?<\|/det\|>",
        "",
        text,
        flags=re.DOTALL
    )

    text = re.sub(
        r"<\|.*?\|>",
        "",
        text,
        flags=re.DOTALL
    )

    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def benchmark_deepseek_ocr2(
    df,
    model,
    tokenizer,
    raw_output_dir="/kaggle/working/deepseek_ocr2_raw",
    prompt="<image>\n<|grounding|>Convert the document to markdown. ",
    base_size=1024,
    image_size=768,
    crop_mode=True,
):

    os.makedirs(raw_output_dir, exist_ok=True)

    results_df = df.copy()

    # --------------------------------------------------
    # Add result columns
    # --------------------------------------------------

    for col in ["prediction", "runtime", "raw_output", "model"]:
        if col not in results_df.columns:
            results_df[col] = None

    total = len(results_df)

    # --------------------------------------------------
    # Resume
    # --------------------------------------------------

    completed = (
        results_df["prediction"].notna()
        & results_df["prediction"].astype(str).str.strip().ne("")
    )

    already_done = int(completed.sum())

    print(f"Total documents : {total}")
    print(f"Already done    : {already_done}")
    print(f"Remaining       : {total - already_done}")
    print("-" * 60)

    # --------------------------------------------------
    # Progress bar
    # --------------------------------------------------

    pbar = tqdm(
        total=total,
        initial=already_done,
        desc="DeepSeek-OCR-2",
        unit="doc",
        dynamic_ncols=True
    )

    # --------------------------------------------------
    # Benchmark
    # --------------------------------------------------

    for counter, idx in enumerate(results_df.index, start=1):

        if completed.loc[idx]:
            continue

        row = results_df.loc[idx]

        doc_id = str(row["doc_id"])
        csv_image_path = str(row["image"])

        filename = os.path.basename(csv_image_path)

        image_path = os.path.join(
            "/kaggle/input/datasets/manyabhatnagar/inputs/ClinOCR_images",
            filename
        )

        if not os.path.exists(image_path):

            tqdm.write(
                f"ERROR | doc_id={doc_id} | image not found"
            )

            results_df.at[idx, "prediction"] = ""
            results_df.at[idx, "runtime"] = None
            results_df.at[idx, "raw_output"] = None
            results_df.at[idx, "model"] = "DeepSeek-OCR-2"

            continue

        # --------------------------------------------------
        # Raw output
        # --------------------------------------------------

        raw_filename = f"{doc_id}.txt"

        raw_path = os.path.join(
            raw_output_dir,
            raw_filename
        )

        try:

            # --------------------------------------------------
            # Timer
            # --------------------------------------------------

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start = time.perf_counter()

            # --------------------------------------------------
            # Suppress DeepSeek's massive output
            # --------------------------------------------------

            stdout_buffer = io.StringIO()
            stderr_buffer = io.StringIO()

            with redirect_stdout(stdout_buffer), redirect_stderr(stderr_buffer):

                with torch.inference_mode():

                    returned_output = model.infer(
                        tokenizer,
                        prompt=prompt,
                        image_file=image_path,
                        output_path=raw_output_dir,
                        base_size=base_size,
                        image_size=image_size,
                        crop_mode=crop_mode,
                        save_results=False,
                        eval_mode=True,
                    )

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            runtime = time.perf_counter() - start

            # --------------------------------------------------
            # Get prediction
            # --------------------------------------------------

            if returned_output is None:

                raise RuntimeError(
                    "DeepSeek-OCR-2 infer() returned None."
                )

            raw_text = str(returned_output)

            prediction = clean_deepseek_output(raw_text)

            # --------------------------------------------------
            # Save raw output
            # --------------------------------------------------

            with open(
                raw_path,
                "w",
                encoding="utf-8"
            ) as f:
                f.write(raw_text)

            # --------------------------------------------------
            # Store results
            # --------------------------------------------------

            results_df.at[idx, "prediction"] = prediction
            results_df.at[idx, "runtime"] = runtime
            results_df.at[idx, "raw_output"] = raw_filename
            results_df.at[idx, "model"] = "DeepSeek-OCR-2"

            completed.loc[idx] = True

            # --------------------------------------------------
            # UPDATE PROGRESS BAR
            # --------------------------------------------------

            pbar.update(1)

            pbar.set_postfix(
                doc=doc_id,
                time=f"{runtime:.1f}s"
            )

        except Exception as e:

            tqdm.write(
                f"ERROR | doc_id={doc_id} | "
                f"{type(e).__name__}: {e}"
            )

            results_df.at[idx, "prediction"] = ""
            results_df.at[idx, "runtime"] = None
            results_df.at[idx, "raw_output"] = None
            results_df.at[idx, "model"] = "DeepSeek-OCR-2"

    pbar.close()

    # --------------------------------------------------
    # Summary
    # --------------------------------------------------

    successful = (
        results_df["prediction"].notna()
        & results_df["prediction"].astype(str).str.strip().ne("")
    )

    print("-" * 60)

    print(
        f"Benchmark complete: "
        f"{int(successful.sum())}/{total} successful"
    )

    return results_df

In [ ]:
deepseek_results = benchmark_deepseek_ocr2(
    df,
    model,
    tokenizer,
    raw_output_dir="/kaggle/working/deepseek_ocr2_raw"
)

In [ ]:
def save_benchmark_csv(
    df,
    output_path="/kaggle/working/deepseek_ocr2_results.csv"
):
    """
    Save benchmark dataframe as CSV.
    """

    df.to_csv(output_path, index=False)

    print(f"Saved CSV to:")
    print(output_path)

    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")

    return output_path

In [ ]:
save_benchmark_csv(
    deepseek_results,
    "/kaggle/working/deepseek_ocr2_results.csv"
)

In [ ]:
import os
import shutil
from IPython.display import FileLink, display


def zip_and_download_raw_outputs(
    raw_output_dir,
    zip_path="/kaggle/working/deepseek_ocr2_raw.zip"
):
    """
    Zip the raw output directory and display a Kaggle download link.
    """

    raw_output_dir = os.path.abspath(raw_output_dir)
    zip_path = os.path.abspath(zip_path)

    if not os.path.isdir(raw_output_dir):
        raise FileNotFoundError(
            f"Raw output directory does not exist: {raw_output_dir}"
        )

    # Remove old zip if it exists
    if os.path.exists(zip_path):
        os.remove(zip_path)

    # shutil.make_archive expects the filename without extension
    zip_base = os.path.splitext(zip_path)[0]

    shutil.make_archive(
        zip_base,
        "zip",
        root_dir=raw_output_dir
    )

    print(f"Created ZIP:")
    print(zip_path)

    file_size_mb = os.path.getsize(zip_path) / (1024 ** 2)
    print(f"Size: {file_size_mb:.2f} MB")

    display(FileLink(zip_path, result_html_prefix="Download: "))

    return zip_path

In [ ]:
zip_and_download_raw_outputs(
    "/kaggle/working/deepseek_ocr2_raw",
    "/kaggle/working/deepseek_ocr2_raw.zip"
)

In [6]:
!pip install -q "chandra-ocr[hf]"

In [ ]:
device_map = {
    # ==========================================
    # GPU 0
    # ==========================================

    "model.visual": 0,

    # TIED WEIGHTS — KEEP TOGETHER
    "model.language_model.embed_tokens": 0,
    "lm_head": 0,

    # First few language layers
    "model.language_model.layers.0": 0,
    "model.language_model.layers.1": 0,
    "model.language_model.layers.2": 0,
    "model.language_model.layers.3": 0,

    # ==========================================
    # GPU 1
    # ==========================================

    "model.language_model.layers.4": 1,
    "model.language_model.layers.5": 1,
    "model.language_model.layers.6": 1,
    "model.language_model.layers.7": 1,
    "model.language_model.layers.8": 1,
    "model.language_model.layers.9": 1,
    "model.language_model.layers.10": 1,
    "model.language_model.layers.11": 1,
    "model.language_model.layers.12": 1,
    "model.language_model.layers.13": 1,
    "model.language_model.layers.14": 1,
    "model.language_model.layers.15": 1,
    "model.language_model.layers.16": 1,
    "model.language_model.layers.17": 1,
    "model.language_model.layers.18": 1,
    "model.language_model.layers.19": 1,
    "model.language_model.layers.20": 1,
    "model.language_model.layers.21": 1,
    "model.language_model.layers.22": 1,
    "model.language_model.layers.23": 1,
    "model.language_model.layers.24": 1,
    "model.language_model.layers.25": 1,
    "model.language_model.layers.26": 1,
    "model.language_model.layers.27": 1,
    "model.language_model.layers.28": 1,
    "model.language_model.layers.29": 1,
    "model.language_model.layers.30": 1,
    "model.language_model.layers.31": 1,

    "model.language_model.norm": 1,
    "model.language_model.rotary_emb": 1,
}

In [7]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "datalab-to/chandra-ocr-2"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model.processor = processor
model.eval()

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors:   0%|          | 0.00/10.6G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/724 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [15]:
def benchmark_chandra2(
    df,
    model,
    tokenizer=None,
    raw_output_dir="/kaggle/working/chandra2_raw",
    image_dir="/kaggle/input/datasets/manyabhatnagar/inputs/ClinOCR_images",
    prompt="Convert the document to markdown.",
    max_output_tokens=4096,
    max_image_size=768,
):

    import os
    import re
    import time
    import shutil
    import warnings
    import torch

    from PIL import Image
    from bs4 import BeautifulSoup
    from IPython.display import display, HTML

    from chandra.model import BatchInputItem
    from chandra.model.hf import generate_hf

    warnings.filterwarnings("ignore")

    # ==========================================================
    # DELETE EVERYTHING FROM PREVIOUS RUN
    # ==========================================================

    if os.path.exists(raw_output_dir):
        shutil.rmtree(raw_output_dir)

    os.makedirs(raw_output_dir, exist_ok=True)

    old_csv = "/kaggle/working/chandra2_results.csv"

    if os.path.exists(old_csv):
        os.remove(old_csv)

    old_zip = "/kaggle/working/chandra2_raw.zip"

    if os.path.exists(old_zip):
        os.remove(old_zip)

    # ==========================================================
    # DISPLAY HELPER
    # ==========================================================

    def show_status(message):

        display(
            HTML(
                f"""
                <pre style="
                    font-family: monospace;
                    font-size: 14px;
                    margin: 4px 0;
                    white-space: pre-wrap;
                ">{message}</pre>
                """
            )
        )

    # ==========================================================
    # INITIAL MESSAGE
    # ==========================================================

    total = len(df)

    show_status(
        "==================================================\n"
        "CHANDRA-OCR-2 BENCHMARK STARTING\n"
        "==================================================\n"
        f"Total documents: {total}\n"
        f"Raw output directory cleared: {raw_output_dir}\n"
        "Previous CSV deleted\n"
        "Previous ZIP deleted\n"
        "=================================================="
    )

    # ==========================================================
    # COPY DATAFRAME
    # ==========================================================

    results_df = df.copy()

    for col in [
        "prediction",
        "runtime",
        "raw_output",
        "model"
    ]:
        if col not in results_df.columns:
            results_df[col] = None

    # ==========================================================
    # LOOP
    # ==========================================================

    successful = 0
    failed = 0

    for counter, idx in enumerate(
        results_df.index,
        start=1
    ):

        row = results_df.loc[idx]

        doc_id = str(
            row["doc_id"]
        )

        filename = os.path.basename(
            str(row["image"])
        )

        image_path = os.path.join(
            image_dir,
            filename
        )

        # ------------------------------------------------------
        # SHOW CURRENT FILE
        # ------------------------------------------------------

        show_status(
            "--------------------------------------------------\n"
            f"[{counter}/{total}] Processing\n"
            f"doc_id     : {doc_id}\n"
            f"image      : {filename}\n"
            f"completed  : {successful}\n"
            f"remaining  : {total - successful}\n"
            "--------------------------------------------------"
        )

        # ------------------------------------------------------
        # CHECK IMAGE
        # ------------------------------------------------------

        if not os.path.exists(image_path):

            failed += 1

            results_df.at[
                idx,
                "prediction"
            ] = ""

            results_df.at[
                idx,
                "runtime"
            ] = None

            results_df.at[
                idx,
                "raw_output"
            ] = None

            results_df.at[
                idx,
                "model"
            ] = "Chandra-OCR-2"

            show_status(
                f"[{counter}/{total}] ERROR\n"
                f"doc_id: {doc_id}\n"
                f"Image not found:\n"
                f"{image_path}"
            )

            continue

        # ------------------------------------------------------
        # RAW OUTPUT PATH
        # ------------------------------------------------------

        raw_filename = f"{doc_id}.txt"

        raw_path = os.path.join(
            raw_output_dir,
            raw_filename
        )

        try:

            # ==================================================
            # LOAD IMAGE
            # ==================================================

            img = Image.open(
                image_path
            ).convert("RGB")

            original_size = img.size

            # ==================================================
            # RESIZE
            # ==================================================

            if max(img.size) > max_image_size:

                img.thumbnail(
                    (
                        max_image_size,
                        max_image_size
                    ),
                    Image.Resampling.LANCZOS
                )

            # ==================================================
            # SHOW THAT INFERENCE IS STARTING
            # ==================================================

            show_status(
                f"[{counter}/{total}] RUNNING INFERENCE\n"
                f"doc_id       : {doc_id}\n"
                f"original     : {original_size}\n"
                f"resized      : {img.size}\n"
                f"completed    : {successful}\n"
                f"remaining    : {total - successful}\n"
                "\n"
                ">>> Chandra inference started...\n"
                ">>> This may take a while."
            )

            # ==================================================
            # BATCH
            # ==================================================

            batch = [
                BatchInputItem(
                    image=img,
                    prompt=prompt,
                    prompt_type=None
                )
            ]

            # ==================================================
            # TIMER
            # ==================================================

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            start = time.perf_counter()

            # ==================================================
            # INFERENCE
            # ==================================================

            with torch.inference_mode():

                result = generate_hf(
                    batch,
                    model,
                    max_output_tokens=max_output_tokens
                )

            if torch.cuda.is_available():
                torch.cuda.synchronize()

            runtime = (
                time.perf_counter()
                - start
            )

            # ==================================================
            # CHECK RESULT
            # ==================================================

            if (
                result is None
                or len(result) == 0
            ):
                raise RuntimeError(
                    "Chandra returned no result."
                )

            raw_text = str(
                result[0].raw
            )

            # ==================================================
            # REMOVE IMAGE HTML
            # ==================================================

            raw_text_no_images = re.sub(
                r"<img\b[^>]*>",
                "",
                raw_text,
                flags=re.IGNORECASE
            )

            # ==================================================
            # BR → NEWLINE
            # ==================================================

            raw_text_no_images = re.sub(
                r"<br\s*/?>",
                "\n",
                raw_text_no_images,
                flags=re.IGNORECASE
            )

            # ==================================================
            # EXTRACT ONLY TEXT
            # ==================================================

            soup = BeautifulSoup(
                raw_text_no_images,
                "html.parser"
            )

            prediction = soup.get_text(
                separator="\n"
            )

            lines = []

            for line in prediction.splitlines():

                line = line.strip()

                if line:
                    lines.append(line)

            prediction = "\n".join(lines)

            prediction = re.sub(
                r"\n{3,}",
                "\n\n",
                prediction
            ).strip()

            # ==================================================
            # SAVE RAW OUTPUT
            # ==================================================

            with open(
                raw_path,
                "w",
                encoding="utf-8"
            ) as f:
                f.write(raw_text)

            # ==================================================
            # SAVE RESULTS
            # ==================================================

            results_df.at[
                idx,
                "prediction"
            ] = prediction

            results_df.at[
                idx,
                "runtime"
            ] = runtime

            results_df.at[
                idx,
                "raw_output"
            ] = raw_filename

            results_df.at[
                idx,
                "model"
            ] = "Chandra-OCR-2"

            successful += 1

            # ==================================================
            # SHOW SUCCESS IMMEDIATELY
            # ==================================================

            show_status(
                f"[{counter}/{total}] ✓ DONE\n"
                f"doc_id       : {doc_id}\n"
                f"runtime      : {runtime:.2f} sec\n"
                f"prediction   : {len(prediction):,} chars\n"
                f"raw output   : {raw_filename}\n"
                f"\n"
                f"COMPLETED    : {successful}/{total}\n"
                f"REMAINING    : {total - successful}\n"
            )

        # ======================================================
        # ERROR
        # ======================================================

        except Exception as e:

            failed += 1

            results_df.at[
                idx,
                "prediction"
            ] = ""

            results_df.at[
                idx,
                "runtime"
            ] = None

            results_df.at[
                idx,
                "raw_output"
            ] = None

            results_df.at[
                idx,
                "model"
            ] = "Chandra-OCR-2"

            show_status(
                f"[{counter}/{total}] ✗ ERROR\n"
                f"doc_id: {doc_id}\n"
                f"{type(e).__name__}: {e}\n"
                f"\n"
                f"COMPLETED: {successful}/{total}\n"
                f"FAILED   : {failed}"
            )

    # ==========================================================
    # FINAL
    # ==========================================================

    show_status(
        "==================================================\n"
        "CHANDRA-OCR-2 BENCHMARK COMPLETE\n"
        "==================================================\n"
        f"Total      : {total}\n"
        f"Successful : {successful}\n"
        f"Failed     : {failed}\n"
        "=================================================="
    )

    return results_df

In [16]:
chandra2_results = benchmark_chandra2(
    df,
    model,
    tokenizer=None
)

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


... (182 outputs left out)

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


In [17]:
output_csv = "/kaggle/working/chandra2_results.csv"

chandra2_results.to_csv(
    output_csv,
    index=False
)

print(f"Saved CSV to: {output_csv}")
print(f"Rows: {len(chandra2_results)}")
print(f"Columns: {len(chandra2_results.columns)}")
print(os.path.exists(output_csv))

Saved CSV to: /kaggle/working/chandra2_results.csv
Rows: 192
Columns: 12
True


In [18]:
import shutil
import os

raw_dir = "/kaggle/working/chandra2_raw"
zip_base = "/kaggle/working/chandra2_raw"

zip_path = shutil.make_archive(
    zip_base,
    "zip",
    raw_dir
)

print(f"Created: {zip_path}")
print(f"Exists: {os.path.exists(zip_path)}")
print(f"Size: {os.path.getsize(zip_path) / 1024**2:.2f} MB")

Created: /kaggle/working/chandra2_raw.zip
Exists: True
Size: 0.24 MB


In [24]:
import transformers
import PIL
print("PyTorch       :", torch.__version__)
print("Transformers  :", transformers.__version__)
print("Pillow        :", PIL.__version__)

PyTorch       : 2.10.0+cu128
Transformers  : 5.15.0
Pillow        : 11.3.0
